# Predict poster quality from numeric features

Train an AutoGluon regression model for `overall_design_score` using character count, text density, aspect ratio, or other measured numeric features. The existing splits stay fixed: **153 train, 33 validation, 33 test**. Images and identifiers are excluded from model inputs.

This follows the example notebook's install → load → check → fit → evaluate → save structure. There is no augmentation. Overall design score is not a direct “needs more text” label.


## 1. Find the repository and install packages

Use Python 3.10–3.13 (3.12 recommended). Run locally from the repository or its `notebooks` folder. In Colab, the next cell clones the repository when needed. On macOS, install OpenMP with `brew install libomp` for LightGBM.


In [ ]:
from pathlib import Path
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "scripts/train_numeric.py").exists()), None)
if ROOT is None and Path("/content").exists():
    ROOT = Path("/content/PosterScorer")
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/jackstev-edu/PosterScorer.git", str(ROOT)],
            check=True,
        )
if ROOT is None or not (ROOT / "scripts/train_numeric.py").exists():
    raise FileNotFoundError("Open this notebook from the PosterScorer repository.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-training.txt")],
    check=True,
)
sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)


## 2. Configure features and files

Copy `data/posteriq/features_template.csv` to `data/posteriq/features.csv` and fill `character_count` and `text_density`. Keep the supplied IDs unchanged. In Colab, upload the completed file into the displayed folder using the Files panel.

- **character_count:** count of non-whitespace characters.
- **text_density:** union area of text regions ÷ total image area, between 0 and 1.
- **aspect_ratio:** width ÷ height; already measured in the template.

Use consistent measurement rules across all posters. Leave unknown values blank; use zero only for a confirmed absence of text. Optional `word_count`, `width_px`, and `height_px` can be added to `FEATURE_COLUMNS`. Text measurements are intentionally not invented.


In [ ]:
import pandas as pd
from IPython.display import display
from scripts.train_numeric import DEFAULT_FEATURES, TARGET, load_splits, train_model

DATA_DIR = ROOT / "data/posteriq"
FEATURES_CSV = DATA_DIR / "features.csv"
FEATURE_COLUMNS = list(DEFAULT_FEATURES)
OUTPUT_DIR = ROOT / "models/numeric"
TIME_LIMIT = 120
NUM_CPUS = 2
RUN_FINAL_TEST = False  # Set True only after choosing your final configuration.

print("Put your completed feature table here:", FEATURES_CSV)
display(pd.read_csv(DATA_DIR / "features_template.csv").head())


## 3. Load and check the data

The loader joins measurements to the existing splits by ID, checks split separation, and validates the selected numeric columns. Some missing measurements can be imputed by AutoGluon, but an entirely unmeasured training feature is rejected. IDs, filenames, and scores are not predictors.

For an initial geometry-only run, set `FEATURE_COLUMNS = ["aspect_ratio", "width_px", "height_px"]` and use a separate output directory.


In [ ]:
if not FEATURES_CSV.exists():
    raise FileNotFoundError(
        f"Copy {DATA_DIR / 'features_template.csv'} to {FEATURES_CSV}, "
        "fill the text measurements, then rerun this cell."
    )
splits = load_splits(DATA_DIR, FEATURES_CSV, FEATURE_COLUMNS)
display(pd.DataFrame({"split": list(splits), "posters": [len(frame) for frame in splits.values()]}))
display(splits["train"][[TARGET] + FEATURE_COLUMNS].describe().T)
display(splits["train"][FEATURE_COLUMNS].isna().sum().rename("missing_training_values"))


## 4. Fit the regression models

Train uses the 153 training rows; validation is supplied as AutoGluon's `tuning_data` for model selection. Candidates include linear regression, random forest, extra trees, and LightGBM, plus a weighted ensemble. Only the selected numeric columns reach the model. The time limit is a training budget; total elapsed time may be longer.

Use a new `OUTPUT_DIR` for another experiment. Keep `RUN_FINAL_TEST = False` while choosing features and settings. Fixed splits aid comparison; exact training results can still vary by runtime.


In [ ]:
predictor, report = train_model(
    features_csv=FEATURES_CSV,
    feature_columns=FEATURE_COLUMNS,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    time_limit=TIME_LIMIT,
    num_cpus=NUM_CPUS,
    evaluate_test=RUN_FINAL_TEST,
)


## 5. Review validation and optional final test results

RMSE and MAE are in score units; lower is better. Higher R² is better and negative values are possible. The baseline always predicts the training mean score. AutoGluon's leaderboard uses negative RMSE, so closer to zero is better.

Validation is used for selection. Reserve the 33 test posters for a final estimate; do not use test results to choose features.


In [ ]:
display(pd.DataFrame(report["validation"]).T)
display(pd.read_csv(OUTPUT_DIR / "validation_leaderboard.csv"))
if "test" in report:
    print("Final test results:")
    display(pd.DataFrame(report["test"]).T)
else:
    print("Test scoring is disabled. Enable RUN_FINAL_TEST only for the final experiment.")


## 6. Load the saved model and predict

The training function saves `model/`, `metrics.json`, `validation_predictions.csv`, and `validation_leaderboard.csv`. Test evaluation also writes `test_predictions.csv`. Keep the entire model directory. In Colab, download the output directory before ending the runtime.

This inference example uses a real validation row. For a new poster, supply the same measured numeric columns and measurement conventions.


In [ ]:
from autogluon.tabular import TabularPredictor

saved_predictor = TabularPredictor.load(str(OUTPUT_DIR / "model"))
inputs = splits["validation"].loc[:, FEATURE_COLUMNS].head(1)
result = splits["validation"][["id", TARGET]].head(1).copy()
result["predicted_score"] = saved_predictor.predict(inputs)
display(result)
print("Saved outputs:", OUTPUT_DIR)


With only 219 independent posters, compare a modest feature set against the baseline. Score predictions do not establish that adding text would improve a poster.

See [the training guide](../docs/numeric-training.md) and the official [AutoGluon 1.5 fit API](https://auto.gluon.ai/1.5/api/autogluon.tabular.TabularPredictor.fit.html).
